In [1]:
#%%capture
#!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.0/spark-sql-kafka-0-10_2.12-3.5.0.jar"
#!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-streaming-kafka-0-10_2.12/3.5.0/spark-streaming-kafka-0-10_2.12-3.5.0.jar"

In [2]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.5.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 pyspark-shell'

In [3]:
!pip install mistralai==1.9.7

In [4]:
#Install mistral from github

In [5]:
#!git clone https://github.com/mistralai/client-python.git
#!git clone --branch v1.9.7 --depth 1 https://github.com/mistralai/client-python.git
#cd client-python
#!pip install -e /home/jovyan/client-python/

In [6]:
import pandas as pd
import os
import pickle
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("OCR").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [7]:
# 1. Read Stream from Kafka
raw_stream = spark \
  .readStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "192.168.1.118:8097") \
  .option("subscribe", "input") \
  .option("startingOffsets", "latest") \
  .load()

raw_stream.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [8]:
processed_data = raw_stream.select(
    col("key").cast(StringType()).alias("filename"),
    col("value").alias("png_data")
)

In [9]:
#import requests
from pyspark.sql import Row
from mistralai import DocumentURLChunk, ImageURLChunk, TextChunk
import json
import base64
import os
import time
# ---------------------

# Initialize Mistral client with API key
from mistralai import Mistral
api_key = 'Zs7BIWkcpcsPfI2fgRYkWdbTUSb8FJR4'
client = Mistral(api_key=api_key)

In [10]:
def call_mistral_ocr(row: Row):

    filename = row['filename']
    png_bytes = row['png_data']
    
    # Encode image as base64 for API
    encoded = base64.b64encode(png_bytes).decode()
    base64_data_url = f"data:image/jpeg;base64,{encoded}"
    #print(base64_data_url)
    
    # Process image with OCR
    image_response = client.ocr.process(
        document=ImageURLChunk(image_url=base64_data_url),
        model="mistral-ocr-latest"
    )

    #return filename, image_response, base64_data_url
    # Convert response to JSON
    response_dict = json.loads(image_response.model_dump_json())
    
    json_string = json.dumps(response_dict["pages"][0]["markdown"], indent=4)
    
    return filename, json_string

In [11]:
def call_mistral_8b_latest(row: Row):

    filename = row['filename']
    png_bytes = row['png_data']
    
    # Encode image as base64 for API
    encoded = base64.b64encode(png_bytes).decode()
    base64_data_url = f"data:image/jpeg;base64,{encoded}"
    #print(base64_data_url)
    
    # Process image with OCR
    image_response = client.ocr.process(
        document=ImageURLChunk(image_url=base64_data_url),
        model="mistral-ocr-latest"
    )

    image_ocr_markdown = image_response.pages[0].markdown

    chat_response = client.chat.complete(
        #model="ministral-8b-latest",
        model = "pixtral-12b-latest",
        messages=[
            {
                "role": "user",
                "content": [
                    TextChunk(
                        text=(
                            f"This is image's OCR in markdown:\n\n{image_ocr_markdown}\n.\n"
                            "Convert this into a sensible structured json response. "
                            "The output should be strictly be json with no extra commentary"
                        )
                    ),
                ],
            }
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )

    usage = chat_response.usage

    response_dict = json.loads(chat_response.choices[0].message.content)
    
    return filename, json.dumps(response_dict, indent=4), usage.prompt_tokens, usage.completion_tokens, usage.total_tokens

In [12]:
# Still got error.

In [13]:
!pip install jiwer

In [14]:
from jiwer import cer, wer

def evaluate_ocr_accuracy(ground_truth: str, ocr_text: str):
    """Calculate CER and WER between ground truth and OCR output."""
    character_error_rate = cer(ground_truth, ocr_text)
    word_error_rate = wer(ground_truth, ocr_text)

    #print(f"CER: {character_error_rate:.4f} ({character_error_rate*100:.2f}%)")
    #print(f"WER: {word_error_rate:.4f} ({word_error_rate*100:.2f}%)")
    return character_error_rate, word_error_rate


In [15]:
def process_batch(df, batch_id):
    """
    Function executed for every micro-batch of the Structured Stream.
    Collects the rows and processes them with the OCR function.
    """

    start_time_full = time.time()
    
    # ⚠️ WARNING: .collect() brings ALL data to the driver program. 
    # Use for testing/low-volume only. For high-volume, consider a custom Kafka Connect Sink.
    rows = df.collect()
    
    results = []
    data_dict = {}
    count = 1
    
    for row in rows:
      
        print("row id = ",count)

        filename, ocr_text = call_mistral_ocr(row)
             
        if filename is None:
            filename = f"image_{batch_id}"

        # 1. Convert string back to a dictionary
        data_dict["filename"] = filename
        
        data_dict["message"] = ocr_text
        
        ground_truth_path = "/home/jovyan/g1.txt"

        if ground_truth_path and os.path.exists(ground_truth_path):

            with open(ground_truth_path, 'r', encoding='utf-8') as f:
                ground_truth = json.load(f) if ground_truth_path.endswith('.json') else f.read().strip()

            char_err, word_err = evaluate_ocr_accuracy(ground_truth, ocr_text) #json.dumps(ocr_text, ensure_ascii=False))
            data_dict['char_err'] = char_err
            data_dict['word_err'] = word_err

        
        results.append((filename, data_dict))        
        print(f"OCR Result for {results}") # Log first 80 chars
        count = count+1

    schema = StructType([
        StructField("filename", StringType(), False),
        StructField("ocr_text", StringType(), False) 
    ])

    if results:
        
        results_df = spark.createDataFrame(results, schema)

        # 3. Write results to JSON files (The Solution)
        # Use the batch_id to create a unique subdirectory for the JSON files
        batch_output_path = os.path.join("/home/jovyan/json", f"batch_{batch_id}")

        results_df.write \
            .format("json") \
            .mode("overwrite") \
            .save(batch_output_path)
        
        print(f"Successfully wrote OCR results for Batch {batch_id} to: {batch_output_path}")
        
        end_time_full = time.time()
        
        total_duration = end_time_full - start_time_full
        
        print(f"   > Total Function Duration: {total_duration:.4f} seconds")
    

In [ ]:
query = processed_data.writeStream \
    .foreachBatch(process_batch) \
    .start() 

query.awaitTermination()

row id =  1
CER: 0.0048 (0.48%)
WER: 0.0435 (4.35%)
OCR Result for [('receipt1.png', {'filename': 'receipt1.png', 'message': '"PLACE FACE UP ON DASH\\nCITY OF PALO ALTO\\nNOT VALID FOR\\nONSTREET PARKING\\n\\nExpiration Date/Time\\n11:59 PM\\nAUG 19, 2024\\n\\nPurchase Date/Time: 01:34pm Aug 19, 2024\\nTotal Due: $15.00\\nTotal Paid: $15.00\\nTicket #: 00005883\\nS/N #: 520117260957\\nSetting: Permit Machines\\nMach Name: Civic Center\\nRate: Daily Parking\\nPmt Type: CC (Swipe)\\n\\n#*** -1224, Visa\\nDISPLAY FACE UP ON DASH\\n\\nPERMIT EXPIRES\\nAT MIDNIGHT"', 'char_err': 0.004819277108433735, 'word_err': 0.043478260869565216})]
Successfully wrote OCR results for Batch 1 to: /home/jovyan/json/batch_1
   > Total Function Duration: 6.4149 seconds


In [ ]:
for q in spark.streams.active:
    q.stop()

In [ ]:
#Mistral tutorial https://colab.research.google.com/github/mistralai/cookbook/blob/main/mistral/ocr/structured_ocr.ipynb#scrollTo=po7Cukllt8za